In [ ]:
import os


# ---------- init a cluster with some data
#  (so that we can use manual commands)

# List the current directory
#print(os.listdir(.))


In [ ]:
%%bash
sudo systemctl stop abrtd
sudo sysctl kernel.core_pattern=core.%e.%p
ulimit -c unlimited


In [ ]:
# Ask the user to select a digit in the range 0 to 9
k = int(input("Please select a digit in the range 0 to 9: "))

# Check if the digit is within the valid range
if 0 <= k <= 9:
    print(f"will be running in k{k}")
else:
    print("Invalid selection. Please select a digit between 0 and 9.")
    exit(1)

from pathlib import Path

wd = Path(f"/home/rfriedma/src/k{k}/ceph/build")
os.chdir(wd)
%env CEPH_JTEST_ROOT=/home/rfriedma/src/k{k}/ceph/build
!echo $CEPH_JTEST_ROOT > /tmp/jpath


In [ ]:
!ls -l
!bash -c MGR=0 ls -l



In [ ]:
%%bash
if [ -e /etc/ceph/ceph.conf ]; then
    echo XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
fi


In [ ]:
%%bash

#MDS=0 MGR=1 OSD=3 MON=1 ../src/vstart.sh -n  --without-dashboard --msgr2 -X --memstore -o "memstore_device_bytes=68435456" -o "osd_op_queue=wpq"
#MDS=0 MGR=1 OSD=3 MON=1 ../src/vstart.sh -n --crimson --without-dashboard --msgr2 -X --memstore -o "memstore_device_bytes=68435456" -o "osd_op_queue=wpq"
#MGR=1 MON=1 OSD=1 MDS=0 RGW=0 ../src/vstart.sh -n --crimson --without-dashboard --msgr2 -X --crimson-smp=10 --seastore -o "osd_op_queue=wpq"
#MGR=1 MON=1 OSD=3 MDS=0 RGW=0 ../src/vstart.sh -n --crimson --no-restart --without-dashboard --seastore --seastore-secondary-devs-type ZNS --crimson-smp 10 --debug --msgr2 -X -o "osd_op_queue=wpq"
#MGR=1 MON=1 OSD=3 MDS=0 RGW=0 ../src/vstart.sh -n --crimson --no-restart --without-dashboard --seastore --crimson-smp 10 --debug --msgr2 -X -o "osd_op_queue=wpq"

#MGR=1 MON=1 OSD=0 MDS=0 RGW=1 ../src/vstart.sh -n --crimson --no-restart --without-dashboard --seastore --crimson-smp 1 -o "crimson_seastar_blocked_reactor_notify_ms = 10000" --debug --msgr2 -X -o "osd_op_queue=wpq"

#MGR=1 MON=1 OSD=3 MDS=0 RGW=1 ../src/vstart.sh -n --crimson --no-restart --without-dashboard --seastore --crimson-smp 1 -o "crimson_seastar_blocked_reactor_notify_ms = 10000" --debug --msgr2 -X -o "osd_op_queue=wpq"
whoami

MGR=1 MON=1 OSD=3 MDS=0 RGW=1 ../src/vstart.sh -n --crimson --no-restart --without-dashboard --seastore --seastore-devs /dev/nvme5n1,/dev/nvme6n1 --crimson-smp 24 -o "crimson_seastar_blocked_reactor_notify_ms = 10000" --ms\
gr2 -X -o "osd_op_queue=wpq"

#--no-restart --without-dashboard --seastore --crimson --debug -n


In [ ]:
%%bash

scrtch=to_"`date +%d_%H%M`"
echo $scrtch


sleep 2
bin/ceph -s

#bin/ceph tell osd.* config set debug_osd 20/20

bin/ceph config set global osd_pool_default_pg_autoscale_mode off
sleep 2

#bin/ceph config set osd erasure_code_default_plugin jerasure

# disable rescheduling of the queue due to 'no-scrub' flags
#bin/ceph tell osd.* config set osd_scrub_backoff_ratio 0.9999

# initial set of global scrub scheduling parameters
bin/ceph tell osd.* config set osd_scrub_interval_randomize_ratio 0.1
bin/ceph tell osd.* config set osd_deep_scrub_randomize_ratio 0
bin/ceph tell osd.* config set osd_scrub_min_interval 10000
bin/ceph tell osd.* config set osd_scrub_max_interval 200000
bin/ceph tell osd.* config set osd_deep_scrub_interval 600000


#PL1 is of size 3

# bin/ceph osd pool create pl1 8 8
# #bin/ceph osd pool autoscale-status
# sleep 1
# pl1_num=`bin/ceph osd pool stats pl1 | sed -n -r -e 's/.*id[^0-9]*([0-9]+)$/\1/p'`
# echo $pl1_num
# bin/ceph osd pool set pl1 size 3
# bin/ceph osd pool set pl1 min_size 3
# bin/ceph osd pool set pl1 pg_autoscale_mode off
# bin/ceph osd pool stats
# bin/ceph osd pool set pl1 noscrub 0
# bin/ceph osd pool set pl1 nodeep-scrub 0
# sleep 2
# 
# bin/rados bench -p pl1 -t 1 1 write -b 4096 --max-objects 8  --no-cleanup; 
# bin/rados bench -p pl1 1 write -b 4096 --max-objects 128 --show-time --no-cleanup --run-name eeeee
# 
# #PL2
# 
# bin/ceph osd pool create pl2 8 8
# bin/ceph osd pool set pl2 size 3
# bin/ceph osd pool set pl2 min_size 3
# bin/ceph osd pool set pl2 pg_autoscale_mode off
# bin/ceph osd pool stats
# bin/ceph osd pool set pl2 noscrub 0
# bin/ceph osd pool set pl2 nodeep-scrub 0
# sleep 2
# 
# bin/rados bench -p pl2 -t 1 1 write -b 4096 --max-objects 8  --no-cleanup; 
# bin/rados bench -p pl2 1 write -b 4096 --max-objects 128 --show-time --no-cleanup --run-name eeeee
# sleep 1

# bin/ceph tell osd.* config set debug_osd 20/20
bin/ceph tell osd.* config set debug_osd 5/5
#bin/ceph tell osd.0 dump_scrubs --format=json-pretty > /tmp/ds_00.json
#bin/ceph tell osd.1 dump_scrubs --format=json-pretty > /tmp/ds_01.json
#bin/ceph tell osd.2 dump_scrubs --format=json-pretty > /tmp/ds_02.json


In [ ]:
%%bash
echo '-------->' $CEPH_JTEST_ROOT
jp=$CEPH_JTEST_ROOT
if [ -z $CEPH_JTEST_ROOT ]; then
    echo "CEPH_JTEST_ROOT is not set"
    [[ -f /tmp/jpath ]] && jp=`cat /tmp/jpath` || jp='.'
    echo '-------->' $jp
    %env CEPH_JTEST_ROOT=$jp
fi
cd $jp


In [ ]:
%%bash

file_path="/tmp/jpath"
if [ -f "$file_path" ]; then
        file_contents=$(cat "$file_path")
        echo "File contents read into variable."
else
        echo "File does not exist."
fi

In [ ]:
%%bash

# dumping the counters
function metrics_dump {
  fname=${1:-/tmp/metrics_dump.json}
  msg=${2:-"dumping metrics"}
  echo "$msg" >> "$fname"
  bin/ceph tell osd.0 dump_metrics --format=json-pretty >> "$fname"
  echo "$msg"
  ceph tell osd.0 dump_metrics 2>/dev/null | jq '[.metrics[] | to_entries[] | select(.key | contains("lazy"))] | from_entries'
  ceph tell osd.0 dump_metrics 2>/dev/null | jq '[.metrics[] | to_entries[] | select(.key | contains("trans_srcs_invalidated"))]'
}

In [ ]:
%%bash

# performing read/write

%%bash
NUM_OSDS=3
export NUM_OSDS
bash ~/src/rebalance_bench.sh


In [ ]:
%%bash

#performance data collection (read/write)

function metrics_dump {
  local dmpfn=${1:-/tmp/metrics_dump.json}
  local msg=${2:-"dumping metrics"}
  echo "$msg" >> "$dmpfn"
  bin/ceph tell osd.0 dump_metrics --format=json-pretty >> "$dmpfn"
  echo "$msg ----------------------------------------------------------------------------------------------------------------------------------------------vvvv"
  ceph tell osd.0 dump_metrics 2>/dev/null | jq '[.metrics[] | to_entries[] | select(.key | contains("lazy"))] | from_entries'
  echo "$msg"
  ceph tell osd.0 dump_metrics 2>/dev/null | jq '[.metrics[] | to_entries[] | select(.key | contains("trans_srcs_invalidated")) | select(.value.srcs | contains("READ")) | select(.value.value != 0)]'
}

#mkdir -p ./phase1_jobfiles
fname=pdump_"`date +%d_%H%M`"
short_fname="/tmp/${fname}_final"
echo $fname

#run without the lazy path
ceph tell osd.0 config set seastore_lazy_read_conflict_detection false
metrics_dump "/tmp/${fname}_N0" "before read/write"
echo "with lazy read conflict detection disabled, before executing the read/write:" >> "$short_fname"
tail -n +3 /tmp/${fname}_N0 | jq '[.metrics[] | to_entries[] | select(.key | contains("trans_srcs_invalidated")) | select(.value.srcs | contains("READ")) | select(.value.value != 0)]' >> "$short_fname"

bash ~/src/matan_scr2.sh

metrics_dump "/tmp/${fname}_N1" "after read/write"
echo "--------------------"
echo "The NO results"
ceph tell osd.0 dump_metrics 2>/dev/null | jq '[.metrics[] | to_entries[] | select(.key | contains("trans_srcs_invalidated")) | select(.value.srcs | contains("READ")) | select(.value.value != 0)]'
echo "with lazy read conflict detection disabled, the number of invalidated reads is:" >> "$short_fname"
tail -n +3 /tmp/${fname}_N1 | jq '[.metrics[] | to_entries[] | select(.key | contains("trans_srcs_invalidated")) | select(.value.srcs | contains("READ")) | select(.value.value != 0)]' >> "$short_fname"


echo $fname

ceph tell osd.0 config set seastore_lazy_read_conflict_detection true
metrics_dump "/tmp/${fname}_Y0" "before read/write"
echo "with lazy read conflict detection enabled, before executing the read/write:" >> "$short_fname"
tail -n +3 /tmp/${fname}_Y0 | jq '[.metrics[] | to_entries[] | select(.key | contains("trans_srcs_invalidated")) | select(.value.srcs | contains("READ")) | select(.value.value != 0)]' >> "$short_fname"

bash ~/src/matan_scr2.sh

metrics_dump "/tmp/${fname}_Y1" "after read/write"
echo "--------------------"
echo "The YES results"
ceph tell osd.0 dump_metrics 2>/dev/null | jq '[.metrics[] | to_entries[] | select(.key | contains("trans_srcs_invalidated")) | select(.value.srcs | contains("READ")) | select(.value.value != 0)]'
echo "with lazy read conflict detection enabled, the number of invalidated reads is:" >> "$short_fname"
tail -n +3 /tmp/${fname}_Y1 | jq '[.metrics[] | to_entries[] | select(.key | contains("trans_srcs_invalidated")) | select(.value.srcs | contains("READ")) | select(.value.value != 0)]' >> "$short_fname"




In [ ]:
%%bash

#cd $CEPH_JTEST_ROOT

bin/ceph tell osd.0 dump_scrubs --format=json-pretty > /tmp/ds_00_b4params.json
bin/ceph tell osd.1 dump_scrubs --format=json-pretty > /tmp/ds_01_b4params.json
bin/ceph tell osd.2 dump_scrubs --format=json-pretty > /tmp/ds_02_b4params.json


# set the scheduling parameters

bin/ceph osd pool set pl1 scrub_min_interval 1
bin/ceph osd pool set pl1 scrub_max_interval 2
bin/ceph osd pool set pl1 deep_scrub_interval 3

#bin/ceph osd pool set pl2 scrub_min_interval 2
#bin/ceph osd pool set pl2 scrub_max_interval 4
#bin/ceph osd pool set pl2 deep_scrub_interval 10

sleep 3

bin/ceph tell osd.0 dump_scrubs --format=json-pretty > /tmp/ds_00.json
bin/ceph tell osd.1 dump_scrubs --format=json-pretty > /tmp/ds_01.json
bin/ceph tell osd.2 dump_scrubs --format=json-pretty > /tmp/ds_02.json





In [ ]:
%%bash

#%cd $CEPH_JTEST_ROOT

# common scrub configs
bin/ceph tell osd.* config set osd_blocked_scrub_grace_period 20
bin/ceph tell osd.* config set osd_stats_update_period_scrubbing 2
bin/ceph tell osd.* config set osd_stats_update_period_not_scrubbing 3
bin/ceph tell osd.* config set osd_scrub_backoff_ratio 0.9999
#bin/ceph tell osd.* config set osd_scrub_interval_randomize_ratio 0.1
#bin/ceph tell osd.* config set osd_deep_scrub_randomize_ratio 0

#bin/ceph tell mgr.$(bin/ceph mgr services | jq -r .mgr) config set mgr_stats_period 2


In [ ]:
%%bash
bin/ceph config set osd reactor_backend_stall_timeout 30000

In [ ]:
%%bash

# list the scrub queue
scrtch=to_"`date +'%H%M%S'`"
echo $scrtch
bin/ceph tell osd.0 dump_scrubs --format=json-pretty
bin/ceph tell osd.1 dump_scrubs --format=json-pretty
bin/ceph tell osd.2 dump_scrubs --format=json-pretty
bin/ceph tell osd.0 dump_scrubs --format=json-pretty > /tmp/ds_0$scrtch.json
bin/ceph tell osd.1 dump_scrubs --format=json-pretty > /tmp/ds_1$scrtch.json
bin/ceph tell osd.2 dump_scrubs --format=json-pretty > /tmp/ds_2$scrtch.json


In [ ]:
%%bash
# set scrub parameters to guarantee slow scrub
bin/ceph tell osd.* config set osd_scrub_sleep "3.0"
bin/ceph tell osd.* config set osd_max_scrubs 1
bin/ceph tell osd.* config set osd_scrub_chunk_max 5
bin/ceph tell osd.* config set osd_shallow_scrub_chunk_max 5


In [ ]:
%%bash

pl1_num=`bin/ceph osd pool stats pl1 | sed -n -r -e 's/.*id[^0-9]*([0-9]+)$/\1/p'`
echo $pl1_num
bin/ceph tell osd.* config set osd_scrub_sleep "3.0"
bin/ceph tell osd.* config set osd_max_scrubs 1
bin/ceph tell osd.* config set osd_scrub_chunk_max 5
bin/ceph tell osd.* config set osd_shallow_scrub_chunk_max 5

bin/ceph tell osd.* config set osd_stats_update_period_scrubbing 2
bin/ceph tell osd.* config set osd_stats_update_period_not_scrubbing 2
#bin/ceph tell mgr.$(bin/ceph mgr services | jq -r .mgr) config set mgr_stats_period 2
sleep 1

# set higher urgency to one of the PGs
bin/ceph tell $pl1_num.7 scrub
bin/ceph tell $pl1_num.3 deep-scrub
bin/ceph tell $pl1_num.1 deep-scrub
bin/ceph tell $pl1_num.6 schedule-deep-scrub
echo '---------------'
bin/ceph tell osd.0 dump_scrubs --format=json-pretty
echo '---------------'
bin/ceph tell osd.1 dump_scrubs --format=json-pretty
echo '---------------'
bin/ceph tell osd.2 dump_scrubs --format=json-pretty
sleep 1
bin/ceph pg dump pgs
echo '---------------'
#bin/ceph tell osd.0 dump_scrubs --format=json-pretty
echo '---------------'
bin/ceph tell osd.1 dump_scrubs --format=json-pretty
echo '---------------'
#bin/ceph tell osd.2 dump_scrubs --format=json-pretty
sleep 2
bin/ceph pg dump pgs
echo '---------------'
bin/ceph tell osd.0 dump_scrubs --format=json-pretty
echo '---------------'
bin/ceph tell osd.1 dump_scrubs --format=json-pretty
echo '---------------'
bin/ceph tell osd.2 dump_scrubs --format=json-pretty
sleep 2
bin/ceph pg dump pgs
echo '---------------'
#bin/ceph tell osd.0 dump_scrubs --format=json-pretty
echo '---------------'
bin/ceph tell osd.1 dump_scrubs --format=json-pretty
echo '---------------'
#bin/ceph tell osd.2 dump_scrubs --format=json-pretty



In [ ]:
%%bash

scrtch=to_"`date +'%H%M%S'`"
echo $scrtch

# list the scrub queue
bin/ceph tell osd.0 dump_scrubs --format=json-pretty > /tmp/ds_0$scrtch.json
bin/ceph tell osd.1 dump_scrubs --format=json-pretty > /tmp/ds_1$scrtch.json
bin/ceph tell osd.2 dump_scrubs --format=json-pretty > /tmp/ds_2$scrtch.json
bin/ceph tell osd.0 dump_scrubs --format=json-pretty
bin/ceph tell osd.1 dump_scrubs --format=json-pretty
bin/ceph tell osd.2 dump_scrubs --format=json-pretty


In [ ]:
raise SystemExit("Stop here")

In [ ]:
%%bash

#cat /tmp/ds_01.json | jq -s '[.[]|.[]|select(.eligible==true)]| sort_by(.overdue,.pgid)' |  head -20

#cat /tmp/ds_01.json | jq -s '[.[]|.[]|select(.eligible==true)|  { pgid, overdue, sched_time } ]' |  head -20

#cat /tmp/ds_01.json | jq -s '[.[]|.[]|select(.eligible==true)|  { pgid, overdue, sched_time } ]' |  head -20

#cat ds_01.json | jq -s '[.[]|.[]|select(.eligible==false) | .overdue as $ov | . += { "ov":$ov } | [.] ] '|  head -20

#cat ds_01.json | jq -s '[.[]|.[]|select(.eligible==false) | .overdue as $ov | . += { "ov":$ov|not } | [.] ] '|  head -20

cat /tmp/ds_01.json | jq -s '[.[]|.[]|select(.eligible==true) | .overdue as $ov | . += { "ov":$ov|not } ]| sort_by(.ov,.sched_time,.pgid) | [.]  '|  head -20

#cat /tmp/ds_01.json | jq -s '[.[]|.[]|select(.eligible==true) | .overdue as $ov | .level as $lvl | . += { "ov":$ov|not, "lvl":$lvl } ]| sort_by(.ov,.sched_time,.pgid,.lvl) | [.]  '|  head -20

cat /tmp/ds_01.json | jq -s '[.[]|.[]|select(.eligible==true) | .overdue as $ov | .level as $lvl |
 . += { "ov":$ov|not, "lvl":$lvl } ]| sort_by(.ov,.sched_time,.pgid,.lvl) | [.]  '|  head -20

# use 'eligible' as just one more sort criteria
echo "========================== --- "
cat /tmp/ds_01.json | jq -s '[.[]|.[]| .eligible as $ripe | .overdue as $ov | .level as $lvl |
 . += { "not_ripe":$ripe|not, "not_ov":$ov|not, "lvl":$lvl } ] | 
 sort_by(.not_ripe, .not_ov,.sched_time,.pgid,.lvl) | [.]  '|  head -100

# now - make the sort an irregular one: if comparing the the targets of one PG, level tramps sched time



# Termination


In [ ]:
%%bash
echo '-------->' $CEPH_JTEST_ROOT
cd $CEPH_JTEST_ROOT
if [ -z $CEPH_JTEST_ROOT ]; then
    echo "CEPH_JTEST_ROOT is not set"
    [[ -f /tmp/jpath ]] && jp=`cat /tmp/jpath` || jp='.'
    echo '-------->' $jp
    cd $jp
    %env CEPH_JTEST_ROOT=$jp
fi

pwd

../src/stop.sh --crimson
sleep 4
../src/stop.sh --crimson
